# Multi-Dataset LSTM Training and Tokenizer Comparison on Google Colab

This notebook trains a baseline **LSTM language model** and compares **word-level**, **character-level**, and **BPE** tokenization using the same project codebase.

Despite the filename, this notebook now supports all four datasets in the project:

- `text8`
- `wikitext-103`
- `enwik8`
- `one-billion-word`

To switch datasets, you only need to change the `DATASET_NAME` variable in the configuration section below.

## What is BPE?

**BPE (Byte Pair Encoding)** is a subword tokenization method.

Instead of treating an entire word as a single token or splitting everything into individual characters, BPE learns frequent subword units such as `play` and `ing`. In practice, it often gives a useful trade-off:

- fewer OOV issues than word-level tokenization
- shorter sequences than character-level tokenization
- more flexibility on rare and morphologically complex words

Important caveat: raw perplexity values across `word`, `char`, and `bpe` are **not perfectly comparable**, because each tokenizer defines a different token space and vocabulary size. Use perplexity together with vocabulary size, sequence length, and training time.

In [ ]:
from pathlib import Path
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    %cd /content
    if not Path("/content/text-preprocess-tokenization").exists():
        !git clone https://github.com/HatakekkSheeshh/text-preprocess-tokenization.git
    %cd /content/text-preprocess-tokenization
else:
    print("This notebook is not running in Colab. Make sure your working directory is the project root.")

PROJECT_ROOT = Path.cwd()
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import json
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from src.datasets.load_data import load
from src.training.train_lstm import LSTMTrainingConfig, train_lstm_language_model

PROJECT_ROOT = Path.cwd()
METRICS_ROOT = PROJECT_ROOT / "outputs" / "metrics" / "lstm"
CHECKPOINT_ROOT = PROJECT_ROOT / "outputs" / "checkpoints" / "lstm"


def dataset_slug(dataset_name: str) -> str:
    return dataset_name.replace("-", "_")


def metrics_path_for(run_name: str) -> Path:
    return METRICS_ROOT / f"{run_name}.json"


def checkpoint_dir_for(run_name: str) -> Path:
    return CHECKPOINT_ROOT / run_name


def load_metrics(run_name: str) -> dict:
    return json.loads(metrics_path_for(run_name).read_text(encoding="utf-8"))


In [ ]:
AVAILABLE_DATASETS = ["text8", "wikitext-103", "enwik8", "one-billion-word"]

# Change this to switch datasets.
DATASET_NAME = "text8"

DATASET_NOTES = {
    "text8": "Fastest to start with. Clean and strongly normalized.",
    "wikitext-103": "Good general-purpose word-level benchmark. More natural raw text than text8.",
    "enwik8": "Character-level modeling is especially natural here because the corpus is byte-oriented.",
    "one-billion-word": "Much heavier than the other datasets. Start with smoke or medium settings first.",
}

assert DATASET_NAME in AVAILABLE_DATASETS, f"Unsupported dataset: {DATASET_NAME}"
print(f"Selected dataset: {DATASET_NAME}")
print(DATASET_NOTES[DATASET_NAME])

# Download/load the selected dataset into data/raw/<dataset_name> if needed.
load(DATASET_NAME)
print(f"{DATASET_NAME} is ready.")

## Configure a single LSTM run

Use this section when you want to train one tokenizer in depth on the selected dataset.

Recommended starting points:

- `text8`: `word` or `bpe`
- `wikitext-103`: `word` or `bpe`
- `enwik8`: `char`
- `one-billion-word`: start with `word` and a smaller preset first

In [ ]:
SINGLE_RUN_PRESETS = {
    "smoke": {
        "sequence_length": 32,
        "batch_size": 8,
        "embedding_dim": 32,
        "hidden_dim": 64,
        "num_layers": 1,
        "dropout": 0.1,
        "epochs": 1,
        "learning_rate": 1e-3,
        "max_train_tokens": 4096,
        "max_validation_tokens": 1024,
        "max_test_tokens": 1024,
        "log_interval": 20,
    },
    "medium": {
        "sequence_length": 128,
        "batch_size": 32,
        "embedding_dim": 128,
        "hidden_dim": 256,
        "num_layers": 2,
        "dropout": 0.2,
        "epochs": 2,
        "learning_rate": 1e-3,
        "max_train_tokens": 300000,
        "max_validation_tokens": 50000,
        "max_test_tokens": 50000,
        "log_interval": 100,
    },
    "full": {
        "sequence_length": 128,
        "batch_size": 64,
        "embedding_dim": 256,
        "hidden_dim": 512,
        "num_layers": 2,
        "dropout": 0.2,
        "epochs": 5,
        "learning_rate": 1e-3,
        "max_train_tokens": None,
        "max_validation_tokens": None,
        "max_test_tokens": None,
        "log_interval": 200,
    },
}

COMPARISON_PRESETS = {
    "smoke": {
        "sequence_length": 32,
        "batch_size": 8,
        "embedding_dim": 32,
        "hidden_dim": 64,
        "num_layers": 1,
        "dropout": 0.1,
        "epochs": 1,
        "learning_rate": 1e-3,
        "max_train_tokens": 4096,
        "max_validation_tokens": 1024,
        "max_test_tokens": 1024,
        "log_interval": 20,
    },
    "medium": {
        "sequence_length": 128,
        "batch_size": 32,
        "embedding_dim": 128,
        "hidden_dim": 256,
        "num_layers": 2,
        "dropout": 0.2,
        "epochs": 2,
        "learning_rate": 1e-3,
        "max_train_tokens": 300000,
        "max_validation_tokens": 50000,
        "max_test_tokens": 50000,
        "log_interval": 100,
    },
}

DEFAULT_SINGLE_TOKENIZER = {
    "text8": "word",
    "wikitext-103": "word",
    "enwik8": "char",
    "one-billion-word": "word",
}

DEFAULT_COMPARISON_TOKENIZERS = {
    "text8": ["word", "char", "bpe"],
    "wikitext-103": ["word", "char", "bpe"],
    "enwik8": ["char", "bpe", "word"],
    "one-billion-word": ["word", "char", "bpe"],
}


def default_vocab_size(dataset_name: str, tokenizer_name: str, *, compare_mode: bool) -> int | None:
    if tokenizer_name == "char":
        return None
    if compare_mode:
        return 10000 if dataset_name == "enwik8" else 20000
    if dataset_name == "one-billion-word":
        return 30000
    if dataset_name == "enwik8":
        return 20000
    return 50000


def default_max_fit_texts(dataset_name: str, *, compare_mode: bool) -> int | None:
    if dataset_name == "one-billion-word":
        return 20000 if compare_mode else 50000
    return None


def build_run_name(dataset_name: str, tokenizer_name: str, preset_name: str, *, compare_mode: bool) -> str:
    dataset_part = dataset_slug(dataset_name)
    if compare_mode:
        return f"colab_compare_{dataset_part}_{tokenizer_name}_{preset_name}"
    return f"colab_{dataset_part}_lstm_{tokenizer_name}_{preset_name}"


def make_config(dataset_name: str, tokenizer_name: str, preset_name: str, *, compare_mode: bool = False) -> LSTMTrainingConfig:
    presets = COMPARISON_PRESETS if compare_mode else SINGLE_RUN_PRESETS
    settings = dict(presets[preset_name])

    # Keep One Billion Word reasonably Colab-friendly by default.
    if dataset_name == "one-billion-word" and not compare_mode and preset_name == "full":
        settings["max_train_tokens"] = 1000000
        settings["max_validation_tokens"] = 100000
        settings["max_test_tokens"] = 100000

    return LSTMTrainingConfig(
        dataset_name=dataset_name,
        tokenizer_name=tokenizer_name,
        sequence_length=settings["sequence_length"],
        embedding_dim=settings["embedding_dim"],
        hidden_dim=settings["hidden_dim"],
        num_layers=settings["num_layers"],
        dropout=settings["dropout"],
        batch_size=settings["batch_size"],
        epochs=settings["epochs"],
        learning_rate=settings["learning_rate"],
        max_vocab_size=default_vocab_size(dataset_name, tokenizer_name, compare_mode=compare_mode),
        max_fit_texts=default_max_fit_texts(dataset_name, compare_mode=compare_mode),
        max_train_tokens=settings["max_train_tokens"],
        max_validation_tokens=settings["max_validation_tokens"],
        max_test_tokens=settings["max_test_tokens"],
        device="cuda",
        run_name=build_run_name(dataset_name, tokenizer_name, preset_name, compare_mode=compare_mode),
        log_interval=settings["log_interval"],
    )


SINGLE_RUN_PROFILE = "full"  # smoke, medium, full
TOKENIZER_NAME = DEFAULT_SINGLE_TOKENIZER[DATASET_NAME]

single_run_config = make_config(DATASET_NAME, TOKENIZER_NAME, SINGLE_RUN_PROFILE, compare_mode=False)
single_run_config

## Train one tokenizer deeply

In [ ]:
single_summary = train_lstm_language_model(single_run_config)
single_summary["test"]

In [ ]:
single_metrics = load_metrics(single_summary["run_name"])

single_row = {
    "dataset": DATASET_NAME,
    "run_name": single_summary["run_name"],
    "tokenizer": single_metrics["tokenizer"]["type"],
    "vocab_size": single_metrics["tokenizer"]["vocab_size"],
    "parameters": single_metrics["model"]["num_parameters"],
    "best_val_loss": round(single_metrics["best_validation_loss"], 4),
    "test_loss": round(single_metrics["test"]["loss"], 4),
    "test_ppl": round(single_metrics["test"]["perplexity"], 4),
    "training_seconds": round(single_metrics["total_training_seconds"], 2),
}

display(pd.DataFrame([single_row]))
print("Metrics file:", metrics_path_for(single_summary["run_name"]))
print("Checkpoint dir:", checkpoint_dir_for(single_summary["run_name"]))

history = single_metrics["history"]
epochs = [item["epoch"] for item in history]
train_losses = [item["train"]["loss"] for item in history]
val_losses = [item["validation"]["loss"] for item in history]
train_ppl = [item["train"]["perplexity"] for item in history]
val_ppl = [item["validation"]["perplexity"] for item in history]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, train_losses, marker="o", label="train")
axes[0].plot(epochs, val_losses, marker="o", label="validation")
axes[0].set_title(f"Loss by Epoch ({DATASET_NAME}, {TOKENIZER_NAME})")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].legend()

axes[1].plot(epochs, train_ppl, marker="o", label="train")
axes[1].plot(epochs, val_ppl, marker="o", label="validation")
axes[1].set_title(f"Perplexity by Epoch ({DATASET_NAME}, {TOKENIZER_NAME})")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Perplexity")
axes[1].legend()

plt.tight_layout()
plt.show()

## Compare word vs char vs BPE

This section runs three experiments with the same LSTM architecture and compares them on the selected dataset.

For `one-billion-word`, the comparison preset is intentionally capped to keep the run manageable on Colab.

In [ ]:
COMPARISON_PROFILE = "medium"  # smoke or medium
COMPARISON_TOKENIZERS = DEFAULT_COMPARISON_TOKENIZERS[DATASET_NAME]

comparison_runs = []

for tokenizer_name in COMPARISON_TOKENIZERS:
    config = make_config(DATASET_NAME, tokenizer_name, COMPARISON_PROFILE, compare_mode=True)
    print(config)
    summary = train_lstm_language_model(config)
    comparison_runs.append(summary)

comparison_runs

In [ ]:
comparison_rows = []

for run in comparison_runs:
    metrics = load_metrics(run["run_name"])
    best_val_ppl = min(item["validation"]["perplexity"] for item in metrics["history"])
    comparison_rows.append(
        {
            "dataset": DATASET_NAME,
            "run_name": run["run_name"],
            "tokenizer": metrics["tokenizer"]["type"],
            "vocab_size": metrics["tokenizer"]["vocab_size"],
            "parameters": metrics["model"]["num_parameters"],
            "val_ppl": round(best_val_ppl, 4),
            "test_ppl": round(metrics["test"]["perplexity"], 4),
            "training_seconds": round(metrics["total_training_seconds"], 4),
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

labels = comparison_df["tokenizer"].tolist()
val_ppl = comparison_df["val_ppl"].tolist()
test_ppl = comparison_df["test_ppl"].tolist()
training_seconds = comparison_df["training_seconds"].tolist()
vocab_sizes = comparison_df["vocab_size"].tolist()

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

axes[0].bar(labels, val_ppl)
axes[0].set_title(f"Validation Perplexity ({DATASET_NAME})")
axes[0].set_ylabel("Perplexity")

axes[1].bar(labels, test_ppl)
axes[1].set_title(f"Test Perplexity ({DATASET_NAME})")

axes[2].bar(labels, training_seconds)
axes[2].set_title(f"Training Time ({DATASET_NAME})")
axes[2].set_ylabel("Seconds")

axes[3].bar(labels, vocab_sizes)
axes[3].set_title(f"Vocabulary Size ({DATASET_NAME})")
axes[3].set_ylabel("Tokens")

plt.tight_layout()
plt.show()

## Suggested report usage

When writing the report, keep these points in mind:

- Compare tokenizers primarily **within the same dataset**.
- Do not claim that character-level tokenization is universally best only because its perplexity may be numerically lower.
- Use perplexity together with vocabulary size, sequence length, model size, and training time.
- For `text8`, expect character-level modeling to be unusually strong because the corpus is heavily normalized.
- For `one-billion-word`, emphasize scalability and computational cost because the dataset is much larger and noisier.

A practical workflow is:

1. Start with a `smoke` or `medium` comparison on a dataset.
2. Pick the most interesting tokenizer settings.
3. Run one deeper experiment with the `full` preset.
4. Export the resulting checkpoints and metrics for later report writing.

In [ ]:
export_dir = PROJECT_ROOT / "colab_export_lstm"
zip_base = PROJECT_ROOT / "colab_export_lstm"

if export_dir.exists():
    shutil.rmtree(export_dir)

(export_dir / "metrics").mkdir(parents=True, exist_ok=True)
(export_dir / "checkpoints").mkdir(parents=True, exist_ok=True)

export_run_names = [single_summary["run_name"]] + [run["run_name"] for run in comparison_runs]

for run_name in export_run_names:
    shutil.copy2(metrics_path_for(run_name), export_dir / "metrics" / metrics_path_for(run_name).name)
    shutil.copytree(checkpoint_dir_for(run_name), export_dir / "checkpoints" / run_name, dirs_exist_ok=True)

manifest = {
    "dataset_name": DATASET_NAME,
    "exported_runs": export_run_names,
}
(export_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=export_dir)
print("Created zip:", zip_path)

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(str(PROJECT_ROOT / "colab_export_lstm.zip"))
else:
    print("Run this notebook on Colab to use the direct download helper.")